# The reproduction gate, in miniature

This notebook regenerates **every** figure of the demo from the f3dasm record
`data/` and nothing else. No pickles, no `.npy` files, no numbers typed by hand.

If a result cannot be redrawn from here, it is not a result — it is an anecdote.


In [1]:
import matplotlib
matplotlib.use("Agg")   # scripted execution: save figures, never show them

import matplotlib.pyplot as plt
import numpy as np
from f3dasm import ExperimentData

from make_data import X_HIGH, X_LOW, true_mean, true_sd

data = ExperimentData.from_file("data")
input_df, output_df = data.to_pandas()
x = input_df["x"].to_numpy(float)
y = output_df["y"].to_numpy(float)
grid = np.linspace(X_LOW, X_HIGH, 400)

print(f"record data/: {len(data)} rows")
print("output columns:", list(output_df.columns))

record data/: 60 rows
output columns: ['y', 'y_pred_baseline', 'sd_baseline', '_source_baseline', 'y_pred_hblr', 'sd_hblr', '_source_hblr', 'y_pred_selected', 'sd_selected', '_source_selected']


## Baseline (written by `baseline.py`)

Plotted from the stored columns `y_pred_baseline` and `sd_baseline` — the
notebook does not re-fit anything here.

In [2]:
order = np.argsort(x)
mu_b = output_df["y_pred_baseline"].to_numpy(float)[order]
sd_b = float(output_df["sd_baseline"].iloc[0])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, s=18, color="#333333", zorder=3, label="training data")
ax.plot(x[order], mu_b, color="#1f77b4", lw=2, label="baseline mean (deg 2)")
ax.fill_between(x[order], mu_b - 2 * sd_b, mu_b + 2 * sd_b, color="#1f77b4",
                alpha=0.20, label=f"baseline $\\pm2$ sd (constant, sd = {sd_b:.1f})")
ax.plot(grid, true_mean(grid) + 2 * true_sd(grid), "k--", lw=1.4,
        label="true $\\pm2$ sd (sd = 0.5 x)")
ax.plot(grid, true_mean(grid) - 2 * true_sd(grid), "k--", lw=1.4)
ax.set_xlabel("speed x [m/s]"); ax.set_ylabel("stopping distance y [m]")
ax.set_title("Baseline: right mean, wrong noise")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig("figures/baseline.png", dpi=150)
print("redrew figures/baseline.png from the record")

redrew figures/baseline.png from the record


---

## Agent sections below

Whoever changes the model or selects its hyperparameters appends cells here,
**between this marker and the final cell**. Those cells may only read columns
that are already in the record — `y_pred_hblr`, `sd_hblr`, `y_pred_selected`,
`sd_selected`, … — plus `data/` itself. They must not read `data_test/`, and
they must not load anything from outside the record.

<!-- AGENT CELLS BELOW THIS LINE -->

---

## What the record remembers

In [3]:
print(f"rows: {len(data)}")
print("input columns :", list(input_df.columns))
print("output columns:", list(output_df.columns))

stamps = [c for c in output_df.columns if c.startswith("_source")]
if stamps:
    print("\nprovenance -- who wrote what:")
    for c in stamps:
        print(f"  {c:22s} {output_df[c].iloc[0]}")
else:
    print("\nno provenance columns yet")

rows: 60
input columns : ['x']
output columns: ['y', 'y_pred_baseline', 'sd_baseline', '_source_baseline', 'y_pred_hblr', 'sd_hblr', '_source_hblr', 'y_pred_selected', 'sd_selected', '_source_selected']

provenance -- who wrote what:
  _source_baseline       baseline
  _source_hblr           modeler
  _source_selected       selector
